# NLP Preprocessing for Next Word Prediction RNN

This notebook performs the NLP preprocessing steps on the cleaned text files from `Cleaned_Text/`:
1. **Splitting**: Divides the 22 episodes into Train (E01-E14), Val (E15-E18), and Test (E19-E22).
2. **Tokenization**: Lowercases and tokenizes sentences, keeping contractions (e.g., `don't`) and punctuation.
3. **Filtering**: Retains only sentences with a length between 5 and 30 words (inclusive).
4. **Vocabulary Building**: Constructs a vocabulary mapping tokens to IDs (incorporating `<pad>`, `<unk>`, `<bos>`, and `<eos>`). Only the training set is used to build the vocabulary to prevent data leakage.
5. **RNN Format Preparation**: Generates two standard datasets:
   - **Sentence/Dialogue Turn Format (Padded)**: Individually padded input-target sequences.
   - **Sliding Window Format (Continuous)**: A continuous sequence sliced into fixed-length inputs and targets.

In [1]:
import os
import re
import json
import glob
from collections import Counter

CLEANED_DIR = "Cleaned_Text"
OUTPUT_DIR = "Preprocessed_Data"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Set hyper-parameters for RNN prep
MIN_WORD_LEN = 5
MAX_WORD_LEN = 30
SEQ_LENGTH = 15      # Sequence length for sliding window
MIN_WORD_FREQ = 3    # Filter out rare words (optional, set to 1 to keep all)

print("NLP Preprocessing Configured.")

NLP Preprocessing Configured.


In [2]:
def tokenize(text):
    # Lowercase and extract word tokens (including contractions) and individual punctuation characters
    text = text.lower()
    tokens = re.findall(r"[a-z0-9]+(?:'[a-z]+)?|[^\n\w\s]", text)
    return [t for t in tokens if t.strip()]

# Example test
sample = "I don't know, Shelly! (train whistle blows) Are we going?"
print("Sample raw text:", sample)
print("Tokenized:", tokenize(sample))

Sample raw text: I don't know, Shelly! (train whistle blows) Are we going?
Tokenized: ['i', "don't", 'know', ',', 'shelly', '!', '(', 'train', 'whistle', 'blows', ')', 'are', 'we', 'going', '?']


In [3]:
# Data splitting rules
train_episodes = [f"S01E{i:02d}" for i in range(1, 15)]
val_episodes = [f"S01E{i:02d}" for i in range(15, 19)]
test_episodes = [f"S01E{i:02d}" for i in range(19, 23)]

def load_split_data(episodes):
    sentences = []
    for ep in episodes:
        filepath = os.path.join(CLEANED_DIR, f"{ep}.txt")
        if not os.path.exists(filepath):
            print(f"Warning: File {filepath} not found. Skipping.")
            continue
        with open(filepath, 'r', encoding='utf-8') as f:
            for line in f:
                line = line.strip()
                if line:
                    sentences.append(line)
    return sentences

train_raw = load_split_data(train_episodes)
val_raw = load_split_data(val_episodes)
test_raw = load_split_data(test_episodes)

print(f"Loaded Raw Sentences: Train={len(train_raw)}, Val={len(val_raw)}, Test={len(test_raw)}")

Loaded Raw Sentences: Train=6072, Val=1731, Test=1839


In [4]:
# Tokenize and apply sentence length constraints (5 to 30 tokens inclusive)
def process_sentences(raw_sentences):
    processed = []
    for s in raw_sentences:
        tokens = tokenize(s)
        # Constraint: Only keep sentences containing between 5 and 30 words/tokens
        if MIN_WORD_LEN <= len(tokens) <= MAX_WORD_LEN:
            processed.append(tokens)
    return processed

train_tokenized = process_sentences(train_raw)
val_tokenized = process_sentences(val_raw)
test_tokenized = process_sentences(test_raw)

print(f"After length filtering ({MIN_WORD_LEN}-{MAX_WORD_LEN} tokens):")
print(f"Train sentences: {len(train_tokenized)} (filtered out {len(train_raw) - len(train_tokenized)})")
print(f"Val sentences: {len(val_tokenized)} (filtered out {len(val_raw) - len(val_tokenized)})")
print(f"Test sentences: {len(test_tokenized)} (filtered out {len(test_raw) - len(test_tokenized)})")

After length filtering (5-30 tokens):
Train sentences: 4700 (filtered out 1372)
Val sentences: 1319 (filtered out 412)
Test sentences: 1428 (filtered out 411)


In [5]:
# Build Vocabulary based on Training split only
word_counter = Counter()
for tokens in train_tokenized:
    word_counter.update(tokens)

# Filter by min frequency if necessary
vocab_list = [word for word, count in word_counter.items() if count >= MIN_WORD_FREQ]

# Add special tokens at indices 0-3
special_tokens = ["<pad>", "<unk>", "<bos>", "<eos>"]
word2idx = {token: idx for idx, token in enumerate(special_tokens)}

for idx, word in enumerate(sorted(vocab_list)):
    word2idx[word] = len(word2idx)

idx2word = {idx: token for token, idx in word2idx.items()}
vocab_size = len(word2idx)

print(f"Vocabulary built: {vocab_size} unique tokens (including special tokens)")
print("Special token indices:", {tok: word2idx[tok] for tok in special_tokens})
print("Sample vocab mapping:", list(word2idx.items())[4:15])

Vocabulary built: 1221 unique tokens (including special tokens)
Special token indices: {'<pad>': 0, '<unk>': 1, '<bos>': 2, '<eos>': 3}
Sample vocab mapping: [('!', 4), ('"', 5), ('$', 6), ('%', 7), ('&', 8), ("'", 9), (',', 10), ('-', 11), ('.', 12), ('00', 13), ('000', 14)]


In [6]:
# Save vocabulary mapping to JSON
vocab_data = {
    "word2idx": word2idx,
    "idx2word": idx2word
}
with open(os.path.join(OUTPUT_DIR, "vocab.json"), 'w', encoding='utf-8') as f:
    json.dump(vocab_data, f, indent=4, ensure_ascii=False)
print(f"Saved vocabulary to {os.path.join(OUTPUT_DIR, 'vocab.json')}")

Saved vocabulary to Preprocessed_Data\vocab.json


In [7]:
# Utility function to map token sequences to integer IDs (using <unk> for unseen tokens)
def map_to_indices(token_seqs):
    unk_id = word2idx["<unk>"]
    return [[word2idx.get(tok, unk_id) for tok in seq] for seq in token_seqs]

train_indexed = map_to_indices(train_tokenized)
val_indexed = map_to_indices(val_tokenized)
test_indexed = map_to_indices(test_tokenized)

print("Example indexed sentence:")
print("Tokens:", train_tokenized[0])
print("Indices:", train_indexed[0])

Example indexed sentence:
Tokens: ["i've", 'always', 'loved', 'trains', '.']
Indices: [525, 54, 624, 1084, 12]


In [8]:
# FORMAT 1: Sentence/Dialogue Level Dataset (with BOS, EOS, and Pad)
bos_id = word2idx["<bos>"]
eos_id = word2idx["<eos>"]
pad_id = word2idx["<pad>"]
unk_id = word2idx["<unk>"]

# Determine max length of sequence in training set (plus BOS/EOS)
max_seq_len = max(len(seq) for seq in train_indexed) + 1  # tokens + 1 special token

def prepare_sentence_dataset(indexed_seqs, max_len):
    inputs = []
    targets = []
    for seq in indexed_seqs:
        inp_seq = [bos_id] + seq
        targ_seq = seq + [eos_id]
        
        pad_len = max_len - len(inp_seq)
        if pad_len > 0:
            inp_seq += [pad_id] * pad_len
            targ_seq += [pad_id] * pad_len
        else:
            inp_seq = inp_seq[:max_len]
            targ_seq = targ_seq[:max_len]
            
        inputs.append(inp_seq)
        targets.append(targ_seq)
    return inputs, targets

train_sent_in, train_sent_targ = prepare_sentence_dataset(train_indexed, max_seq_len)
val_sent_in, val_sent_targ = prepare_sentence_dataset(val_indexed, max_seq_len)
test_sent_in, test_sent_targ = prepare_sentence_dataset(test_indexed, max_seq_len)

print(f"Sentence format dataset built with max_len={max_seq_len}:")
print(f"Train size: {len(train_sent_in)}")
print("Sample Input: ", train_sent_in[0])
print("Sample Target:", train_sent_targ[0])

Sentence format dataset built with max_len=20:
Train size: 4700
Sample Input:  [2, 525, 54, 624, 1084, 12, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
Sample Target: [525, 54, 624, 1084, 12, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]


In [9]:
# FORMAT 2: Sliding Window Format (Continuous stream)
def prepare_sliding_window(indexed_seqs, seq_len):
    flat_stream = []
    for seq in indexed_seqs:
        flat_stream += [bos_id] + seq + [eos_id]
        
    inputs = []
    targets = []
    for i in range(0, len(flat_stream) - seq_len, 1):
        chunk = flat_stream[i : i + seq_len + 1]
        inputs.append(chunk[:-1])
        targets.append(chunk[1:])
        
    return inputs, targets, len(flat_stream)

train_win_in, train_win_targ, train_total_tokens = prepare_sliding_window(train_indexed, SEQ_LENGTH)
val_win_in, val_win_targ, val_total_tokens = prepare_sliding_window(val_indexed, SEQ_LENGTH)
test_win_in, test_win_targ, test_total_tokens = prepare_sliding_window(test_indexed, SEQ_LENGTH)

print(f"Sliding window format dataset built with seq_length={SEQ_LENGTH}:")
print(f"Train: {len(train_win_in)} sequences (flat stream size: {train_total_tokens} tokens)")
print(f"Val: {len(val_win_in)} sequences (flat stream size: {val_total_tokens} tokens)")
print(f"Test: {len(test_win_in)} sequences (flat stream size: {test_total_tokens} tokens)")
print("Sample Input: ", train_win_in[0])
print("Sample Target:", train_win_targ[0])

Sliding window format dataset built with seq_length=15:
Train: 47401 sequences (flat stream size: 47416 tokens)
Val: 13383 sequences (flat stream size: 13398 tokens)
Test: 14442 sequences (flat stream size: 14457 tokens)
Sample Input:  [2, 525, 54, 624, 1084, 12, 3, 2, 531, 351, 10, 528, 700, 1, 3]
Sample Target: [525, 54, 624, 1084, 12, 3, 2, 531, 351, 10, 528, 700, 1, 3, 2]


In [10]:
# Package and save everything to preprocessed_data.json
final_dataset = {
    "metadata": {
        "vocab_size": vocab_size,
        "max_sentence_seq_len": max_seq_len,
        "sliding_window_seq_len": SEQ_LENGTH,
        "min_word_freq": MIN_WORD_FREQ,
        "special_tokens": {
            "pad": pad_id,
            "unk": unk_id,
            "bos": bos_id,
            "eos": eos_id
        }
    },
    "sentence_format": {
        "train": {"inputs": train_sent_in, "targets": train_sent_targ},
        "val": {"inputs": val_sent_in, "targets": val_sent_targ},
        "test": {"inputs": test_sent_in, "targets": test_sent_targ}
    },
    "sliding_window_format": {
        "train": {"inputs": train_win_in, "targets": train_win_targ},
        "val": {"inputs": val_win_in, "targets": val_win_targ},
        "test": {"inputs": test_win_in, "targets": test_win_targ}
    }
}

output_json_path = os.path.join(OUTPUT_DIR, "preprocessed_data.json")
with open(output_json_path, 'w', encoding='utf-8') as f:
    json.dump(final_dataset, f, indent=2, ensure_ascii=False)

print(f"All preprocessed datasets saved successfully to: {output_json_path}")

All preprocessed datasets saved successfully to: Preprocessed_Data\preprocessed_data.json
